[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/42_batchnorm2d.ipynb)

# 🟡 Medium: Implement BatchNorm2d

Implement **Batch Normalization for images**, the version used after every convolution in a CNN, with both **training** and **inference** behavior.

Unlike #07 (inputs `(N, D)`, one statistic per feature), here `x` is `(N, C, H, W)` and there is **one statistic per channel**, computed over the batch **and** all spatial positions:

$$\mu_c = \frac{1}{NHW}\sum_{n,h,w} x_{n,c,h,w}, \qquad \sigma_c^2 = \frac{1}{NHW}\sum_{n,h,w} (x_{n,c,h,w} - \mu_c)^2$$

$$y_{n,c,h,w} = \gamma_c \cdot \frac{x_{n,c,h,w} - \mu_c}{\sqrt{\sigma_c^2 + \epsilon}} + \beta_c$$

### Signature
```python
def my_batch_norm2d(
    x: torch.Tensor,             # (N, C, H, W)
    gamma: torch.Tensor,         # (C,)
    beta: torch.Tensor,          # (C,)
    running_mean: torch.Tensor,  # (C,), updated in place during training
    running_var: torch.Tensor,   # (C,), updated in place during training
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:               # (N, C, H, W)
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm2d`, etc.
- **No Python loops** over channels: reduce over `dim=(0, 2, 3)` and broadcast every `(C,)` tensor as `(1, C, 1, 1)`
- **Normalize** with the **biased** per-channel variance
- **Update** the running stats like `nn.BatchNorm2d`, with the **unbiased** variance for `running_var`:
  - `running_mean = (1 - momentum) * running_mean + momentum * mean`
  - `running_var = (1 - momentum) * running_var + momentum * x.var(dim=(0, 2, 3), unbiased=True)`
  - the Bessel correction counts **every** value in the channel: `N * H * W` samples, not `N`
- In inference mode (`training=False`), normalize with the running stats and leave them unchanged
- Gradients must flow to `x`, `gamma`, `beta`, but **not** into the running stats

> **Builds on:** #07 BatchNorm (same train/eval logic), P2 Broadcasting (`(C,)` → `(1, C, 1, 1)`), P3 Reductions (`channel_stats`: per-channel stats over several dims).


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm2d(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    pass  # Replace this


In [ ]:
# 🧪 Debug
torch.manual_seed(0)
x = torch.randn(4, 3, 8, 8) * 3 + 1
gamma, beta = torch.ones(3), torch.zeros(3)
running_mean, running_var = torch.zeros(3), torch.ones(3)

out = my_batch_norm2d(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out.shape)                            # (4, 3, 8, 8)
print("[Train] Per-channel means:", out.mean(dim=(0, 2, 3)))         # ~0
print("[Train] Per-channel vars: ", out.var(dim=(0, 2, 3), unbiased=False))   # ~1
print("Updated running_mean:", running_mean)
print("Updated running_var: ", running_var)

# Compare with PyTorch
bn = torch.nn.BatchNorm2d(3).train()
ref = bn(x)
print("Matches nn.BatchNorm2d:", torch.allclose(out, ref, atol=1e-4),
      torch.allclose(running_var, bn.running_var, atol=1e-5))


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm2d")
